<a href="https://colab.research.google.com/github/samisileshi/Torrent-drive/blob/main/Torrent_To_Google_Drive_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent To Google Drive Downloader

**Important Note:** To get more disk space:
> Go to Runtime -> Change Runtime and give GPU as the Hardware Accelerator.  You will get around 384GB to download any torrent you want.

### Install libtorrent and Initialize Session

In [2]:
%pip install --ignore-installed libtorrent

import libtorrent as lt

ses = lt.session({'listen_interfaces': '0.0.0.0:6881'})
downloads = []

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 22.4 MB/s  0:00:00


### Mount Google Drive
To stream files we need to mount Google Drive.

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs("/content/drive/My Drive/Torrent", exist_ok=True)

Mounted at /content/drive


### Add From Torrent File
You can run this cell to add more files as many times as you want

In [ ]:
save_path = "/content/drive/My Drive/Torrent"

while True:
    magnet_link = input("Enter Magnet Link Or Type Exit: ")
    if magnet_link.lower() == "exit":
        break
    params = lt.parse_magnet_uri(magnet_link)
    params.save_path = save_path
    downloads.append(ses.add_torrent(params))

### Add From Magnet Link
You can run this cell to add more files as many times as you want

In [ ]:
import time
from IPython.display import display
import ipywidgets as widgets

state_str = [
    "checking",
    "downloading metadata",
    "downloading",
    "finished",
    "seeding",
    "allocating",
    "checking fastresume",
]

layout = widgets.Layout(width="auto")
style = {"description_width": "initial"}
download_bars = [
    widgets.FloatSlider(step=0.01, disabled=True, layout=layout, style=style)
    for _ in downloads
]
display(*download_bars)

while downloads:
    for download, bar in list(zip(downloads, download_bars)):
        s = download.status()
        if not s.is_finished:
            bar.description = " ".join(
                [s.name, str(s.download_rate / 1000), "kB/s", state_str[int(s.state)]]
            )
            bar.value = s.progress * 100
        else:
            ses.remove_torrent(download)
            downloads.remove(download)
            download_bars.remove(bar)
            bar.layout.display = "none"  # bar.close() doesn't work in Colab
            print(s.name, "complete")
    time.sleep(1)

In [ ]:
ses.apply_settings({
    'upload_rate_limit': 0,          # 0 = unlimited
    'download_rate_limit': 0,
    'connections_limit': 500,
    'active_seeds': -1,
    'enable_upnp': False,            # no router to talk to in Colab
    'enable_natpmp': False,
})

### Start Download
Source: https://stackoverflow.com/a/5494823/7957705 and [#3 issue](https://github.com/FKLC/Torrent-To-Google-Drive-Downloader/issues/3) which refers to this [stackoverflow question](https://stackoverflow.com/a/6053350/7957705)

In [ ]:
import time
from IPython.display import display
import ipywidgets as widgets

state_str = [
    "checking",
    "downloading metadata",
    "downloading",
    "finished",
    "seeding",
    "allocating",
    "checking fastresume",
]

layout = widgets.Layout(width="auto")
style = {"description_width": "initial"}
download_bars = [
    widgets.FloatSlider(step=0.01, disabled=True, layout=layout, style=style)
    for _ in downloads
]
display(*download_bars)

while downloads:
    for download, bar in list(zip(downloads, download_bars)):
        s = download.status()
        if not s.is_finished:
            bar.description = " ".join(
                [s.name, str(s.download_rate / 1000), "kB/s", state_str[int(s.state)]]
            )
            bar.value = s.progress * 100
        else:
            ses.remove_torrent(download)
            downloads.remove(download)
            download_bars.remove(bar)
            bar.layout.display = "none"  # bar.close() doesn't work in Colab
            print(s.name, "complete")
    time.sleep(1)